# 확률분포 시 생성기 LoRA 파인튜닝

기준: `Qwen/Qwen2.5-0.5B-Instruct`를 Colab에서 LoRA 파인튜닝하고, adapter와 merged model을 모두 Hugging Face Hub에 업로드한다.

출력 형식은 항상 3줄 한국어 시이며, 프롬프트 모드나 demo fallback은 사용하지 않는다.

In [ ]:
!pip -q install -U transformers accelerate peft datasets trl bitsandbytes huggingface_hub sentencepiece

In [ ]:
from huggingface_hub import login
login()

In [ ]:
BASE_MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
DATA_PATH = 'data/train.jsonl'
ADAPTER_REPO_ID = 'z-unghyun/poem-generator-lora-adapter'
MERGED_REPO_ID = 'z-unghyun/poem-generator-merged'
OUTPUT_DIR = './outputs/poem-generator-lora'

In [ ]:
from datasets import load_dataset

dataset = load_dataset('json', data_files=DATA_PATH, split='train')
dataset

In [ ]:
def format_example(example):
    experience = example['experience'].strip()
    poem = example['poem'].strip()
    return {
        'text': (
            '다음 경험을 바탕으로 3줄의 한국어 시를 써라.\n'
            '조건:\n'
            '- 각 줄은 짧게 쓴다.\n'
            '- 설명문이 아니라 이미지 중심으로 쓴다.\n'
            '- 경험 속 사물이나 감각을 최소 1개 포함한다.\n'
            '- 일상 표현을 살짝 비틀어 언어적 도약을 만든다.\n\n'
            f'경험:\n{experience}\n\n'
            f'시:\n{poem}'
        )
    }

dataset = dataset.map(format_example)
dataset[0]

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'v_proj'],
)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field='text',
    max_seq_length=512,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy='epoch',
    fp16=True,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    peft_config=lora_config,
    tokenizer=tokenizer,
)

In [ ]:
trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

In [ ]:
trainer.model.push_to_hub(ADAPTER_REPO_ID)
tokenizer.push_to_hub(ADAPTER_REPO_ID)

## 다음 단계

학습 후 `training/merge_lora.py` 또는 별도 Colab 셀에서 adapter를 base model에 merge하고 `MERGED_REPO_ID`로 업로드한다. Hugging Face Space에서는 CPU 배포 단순화를 위해 merged model 로드를 우선한다.